In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS gold.dim_date (
    full_date       DATE,
    year_number     INT,
    quarter_number  INT,
    month_number    INT,
    month_name      STRING,
    month_short     STRING,   
    day_of_week     INT,
    day_name        STRING,
    day_short       STRING,  
    day_of_month    INT,
    week_of_year    INT,
    is_weekend      BOOLEAN,
    created_at      TIMESTAMP,
    updated_at      TIMESTAMP
)
USING DELTA
PARTITIONED BY (year_number);

WITH date_bounds AS (
    SELECT
        MIN(order_date) AS min_date,
        MAX(order_date) AS max_date
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
),
final_bounds AS (
    SELECT
        CASE
            WHEN min_date IS NULL THEN add_months(current_date(), -12)
            ELSE add_months(min_date, -12)
        END AS start_date,
        CASE
            WHEN max_date IS NULL THEN add_months(current_date(), 12)
            ELSE add_months(max_date, 12)
        END AS end_date
    FROM date_bounds
),
all_dates AS (
    SELECT explode(sequence(start_date, end_date, interval 1 day)) AS full_date
    FROM final_bounds
),
prepared_data AS (
    SELECT
        full_date,
        year(full_date)                                                   AS year_number,
        quarter(full_date)                                                AS quarter_number,
        month(full_date)                                                  AS month_number,
        date_format(full_date, 'MMMM')                                   AS month_name,
        date_format(full_date, 'MMM')                                    AS month_short,
        dayofweek(full_date)                                              AS day_of_week,
        date_format(full_date, 'EEEE')                                   AS day_name,
        date_format(full_date, 'EEE')                                    AS day_short,
        day(full_date)                                                    AS day_of_month,
        weekofyear(full_date)                                             AS week_of_year,
        CASE WHEN dayofweek(full_date) IN (6, 7) THEN TRUE ELSE FALSE END AS is_weekend,
        current_timestamp()                                               AS updated_at
    FROM all_dates
)
MERGE INTO gold.dim_date AS target
USING prepared_data AS source
ON target.full_date = source.full_date

WHEN MATCHED THEN UPDATE SET
    target.year_number    = source.year_number,
    target.quarter_number = source.quarter_number,
    target.month_number   = source.month_number,
    target.month_name     = source.month_name,
    target.month_short    = source.month_short,
    target.day_of_week    = source.day_of_week,
    target.day_name       = source.day_name,
    target.day_short      = source.day_short,
    target.day_of_month   = source.day_of_month,
    target.week_of_year   = source.week_of_year,
    target.is_weekend     = source.is_weekend,
    target.updated_at     = source.updated_at

WHEN NOT MATCHED THEN INSERT (
    full_date,
    year_number,
    quarter_number,
    month_number,
    month_name,
    month_short,
    day_of_week,
    day_name,
    day_short,
    day_of_month,
    week_of_year,
    is_weekend,
    created_at,
    updated_at
)
VALUES (
    source.full_date,
    source.year_number,
    source.quarter_number,
    source.month_number,
    source.month_name,
    source.month_short,
    source.day_of_week,
    source.day_name,
    source.day_short,
    source.day_of_month,
    source.week_of_year,
    source.is_weekend,
    current_timestamp(),
    current_timestamp()
);